# LeetCode 919 — Complete Binary Tree Inserter bằng Python

Link đề: https://leetcode.com/problems/complete-binary-tree-inserter/ · Độ khó: **Medium** · Chủ đề: Tree, Breadth-First Search, Design, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class CBTInserter:

    def __init__(self, root: Optional[TreeNode]):
        ...

    def insert(self, val: int) -> int:
        ...

    def get_root(self) -> Optional[TreeNode]:
        ...

# Your CBTInserter object will be instantiated and called as such:
# obj = CBTInserter(root)
# param_1 = obj.insert(val)
# param_2 = obj.get_root()
```

Ghi chú: đây là bài **thiết kế lớp** trên **cây nhị phân hoàn chỉnh**. Notebook tự dựng `TreeNode`, hàm dựng cây từ danh sách theo tầng (`build_tree`), hàm xuất cây ra danh sách theo tầng (`to_level`) và bộ chạy chuỗi lệnh `run_ops` (chuyển đối số danh sách thành cây và kết quả `get_root` thành danh sách để so sánh). Điểm cốt lõi: trong cây hoàn chỉnh, nút mới luôn được gắn vào **nút đầu tiên theo thứ tự BFS còn thiếu con**; giữ **hàng đợi các nút còn thiếu con** (hoặc lưu cây dạng mảng theo thứ tự tầng) để chèn trong `O(1)`. Oracle: **mô hình mảng theo tầng** — chèn là `append`, cha của vị trí `k` là `(k − 1) // 2`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cây nhị phân **hoàn chỉnh** (mọi tầng đầy trừ tầng cuối, tầng cuối dồn trái); `insert(v)` thêm nút mới giữ cây hoàn chỉnh và trả về giá trị của **cha** nút mới; `get_root()` trả gốc.
2. Nắm công cụ Python: `collections.deque`, BFS theo tầng, biểu diễn cây hoàn chỉnh bằng **mảng** (chỉ số con `2i+1`, `2i+2`, cha `(i−1)//2`), biểu diễn nhị phân của vị trí để đi từ gốc xuống.
3. Đi từ **BFS tìm vị trí mỗi lần chèn (`O(n)`)** → **hàng đợi các nút còn thiếu con (`O(1)`)** → **mảng các nút theo tầng (`O(1)`)** → **đường đi theo bit của vị trí (`O(log n)`, không cấu trúc phụ)**.
4. Hiểu mấu chốt: nút thứ `k` (đếm từ 1) của cây hoàn chỉnh có cha là nút `⌊k/2⌋`, và là con trái nếu `k` chẵn, con phải nếu `k` lẻ; nên nút mới nằm ở vị trí `n + 1` hoàn toàn xác định.
5. Tránh lỗi kinh điển: bỏ nút khỏi hàng đợi quá sớm (sau khi gắn con **trái**), quên đẩy nút mới vào hàng đợi, khởi tạo hàng đợi sai (đưa vào cả nút đã đủ hai con), đi theo bit sai chiều, và trả về nút thay vì **giá trị** của cha.

## 2. Đề bài & ví dụ

**Cây nhị phân hoàn chỉnh** là cây nhị phân mà mọi tầng, trừ có thể tầng cuối, được **lấp đầy**, và mọi nút ở tầng cuối **dồn về bên trái nhất có thể**.

Thiết kế thuật toán chèn một nút mới vào cây hoàn chỉnh sao cho sau khi chèn cây **vẫn hoàn chỉnh**.

Lớp `CBTInserter`:

- `CBTInserter(TreeNode root)`: khởi tạo cấu trúc với gốc `root` của cây hoàn chỉnh.
- `int insert(int val)`: chèn một `TreeNode` có giá trị `val` để cây vẫn hoàn chỉnh; trả về **giá trị của nút cha** của nút vừa chèn.
- `TreeNode get_root()`: trả về gốc của cây.

**Ví dụ 1:** `["CBTInserter","insert","insert","get_root"]`, đối số `[[[1,2]],[3],[4],[]]` → `[null, 1, 2, [1,2,3,4]]`.

**Ràng buộc:** số nút trong cây `[1, 1000]`; `0 <= Node.val <= 5000`; `root` là cây hoàn chỉnh; `0 <= val <= 5000`; tối đa `10^4` lệnh `insert` và `get_root`.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None là nút trống)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def to_level(root):
    """Xuất cây ra danh sách theo tầng, cắt None thừa ở cuối (giống định dạng kết quả của LeetCode)."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def all_nodes_bfs(root):
    out, q = [], deque([root] if root else [])
    while q:
        node = q.popleft()
        out.append(node)
        for ch in (node.left, node.right):
            if ch:
                q.append(ch)
    return out


def is_complete(root):
    """Kiểm tra cây hoàn chỉnh: duyệt BFS gồm cả None; sau nút None đầu tiên không được còn nút thật."""
    seen_none = False
    q = deque([root])
    while q:
        node = q.popleft()
        if node is None:
            seen_none = True
        else:
            if seen_none:
                return False
            q.append(node.left)
            q.append(node.right)
    return True


def run_ops(cls, ops, args):
    """Bộ chạy chuỗi lệnh kiểu LeetCode cho CBTInserter: đối số khởi tạo là danh sách theo tầng -> dựng cây;
    kết quả get_root (TreeNode) -> danh sách theo tầng để so sánh. Trả về danh sách kết quả (None cho lệnh khởi tạo)."""
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(build_tree(a[0]))
            out.append(None)
        else:
            res = getattr(obj, op)(*a)
            out.append(to_level(res) if op == "get_root" else res)
    return out


example_ops = ["CBTInserter", "insert", "insert", "get_root"]
example_args = [[[1, 2]], [3], [4], []]
example_out = [None, 1, 2, [1, 2, 3, 4]]
print("ví dụ:", example_ops, example_args, "->", example_out)
assert is_complete(build_tree([1, 2, 3, 4, 5, 6])) and not is_complete(build_tree([1, 2, 3, None, 4]))

ví dụ: ['CBTInserter', 'insert', 'insert', 'get_root'] [[[1, 2]], [3], [4], []] -> [None, 1, 2, [1, 2, 3, 4]]


## 3. Kiến thức nền

### 3.1 Cây hoàn chỉnh và chỉ số tầng

Đánh số các nút theo thứ tự **BFS** (từng tầng, trái sang phải) bắt đầu từ `0`: nút `i` có con trái `2i + 1`, con phải `2i + 2`, cha `(i − 1) // 2`. Cây hoàn chỉnh `n` nút chính là các nút có chỉ số `0..n−1` — **không có lỗ hổng**. Đây cũng là cách biểu diễn của **đống nhị phân (binary heap)**.

In [2]:
vals = [10, 20, 30, 40, 50, 60]
for i in range(len(vals)):
    kids = [j for j in (2 * i + 1, 2 * i + 2) if j < len(vals)]
    par = (i - 1) // 2 if i else None
    print(f"chỉ số {i} (giá trị {vals[i]}): con {kids}, cha {par}")

chỉ số 0 (giá trị 10): con [1, 2], cha None
chỉ số 1 (giá trị 20): con [3, 4], cha 0
chỉ số 2 (giá trị 30): con [5], cha 0
chỉ số 3 (giá trị 40): con [], cha 1
chỉ số 4 (giá trị 50): con [], cha 1
chỉ số 5 (giá trị 60): con [], cha 2


### 3.2 Vị trí nút mới và nút cha

Cây có `n` nút (chỉ số `0..n−1`). Nút mới có chỉ số `n`; cha là `(n − 1) // 2`; nó là con **trái** nếu `n` lẻ (`n = 2p + 1`), con **phải** nếu `n` chẵn (`n = 2p + 2`). Quy tắc: nút mới gắn vào nút **đầu tiên theo BFS còn thiếu con**.

### 3.3 Hàng đợi các nút còn thiếu con

Trong cây hoàn chỉnh, các nút còn thiếu con (một con hoặc hai con) tạo thành **đoạn cuối** của thứ tự BFS. Nút đầu tiên trong đoạn đó là cha của nút mới. Sau khi gắn **con trái** nút đó vẫn thiếu một con (còn ở hàng đợi); sau khi gắn **con phải** nút đó đủ hai con và bị loại. Nút mới (không có con) được đẩy vào **cuối** hàng đợi.

### 3.4 Đường đi theo bit

Với vị trí 1-indexed `k = n + 1`, biểu diễn nhị phân của `k` (bỏ bit `1` đầu) cho đường đi từ gốc: bit `0` = trái, bit `1` = phải. Nút thứ `k` nằm đúng ở cuối đường đi đó.

In [3]:
for k in range(1, 10):
    bits = bin(k)[3:]
    print(f"vị trí {k}: nhị phân {bin(k)[2:]:>4} -> đường đi từ gốc {bits or '(gốc)':>4} ({'-'.join('R' if b == '1' else 'L' for b in bits) or 'gốc'})")

vị trí 1: nhị phân    1 -> đường đi từ gốc (gốc) (gốc)
vị trí 2: nhị phân   10 -> đường đi từ gốc    0 (L)
vị trí 3: nhị phân   11 -> đường đi từ gốc    1 (R)
vị trí 4: nhị phân  100 -> đường đi từ gốc   00 (L-L)
vị trí 5: nhị phân  101 -> đường đi từ gốc   01 (L-R)
vị trí 6: nhị phân  110 -> đường đi từ gốc   10 (R-L)
vị trí 7: nhị phân  111 -> đường đi từ gốc   11 (R-R)
vị trí 8: nhị phân 1000 -> đường đi từ gốc  000 (L-L-L)
vị trí 9: nhị phân 1001 -> đường đi từ gốc  001 (L-L-R)


## 4. Ý tưởng nền tảng / trực giác

Cây `[1, 2]` (gốc `1`, con trái `2`). Thêm `3`: nút đầu tiên theo BFS còn thiếu con là gốc `1` (thiếu con phải) ⇒ `3` thành con phải của `1`, trả `1`. Cây `[1, 2, 3]`. Thêm `4`: nút đầu tiên còn thiếu con là `2` (không con nào) ⇒ `4` là con trái của `2`, trả `2`. Cây `[1, 2, 3, 4]` ✓.

```
   1            1            1
  /     ->     / \    ->    / \
 2            2   3        2   3
                          /
                         4
```

Mỗi lần chèn chỉ cần biết "nút đầu tiên còn thiếu con" — thông tin này đổi rất ít sau mỗi lần chèn, nên có thể duy trì thay vì tìm lại.

In [4]:
root = build_tree([1, 2])
q = deque(n for n in all_nodes_bfs(root) if n.left is None or n.right is None)
print("nút còn thiếu con theo BFS:", [n.val for n in q])
for v in (3, 4):
    parent = q[0]
    node = TreeNode(v)
    if parent.left is None:
        parent.left = node
    else:
        parent.right = node
        q.popleft()
    q.append(node)
    print(f"chèn {v}: cha {parent.val}; hàng đợi giờ: {[n.val for n in q]}; cây {to_level(root)}")

nút còn thiếu con theo BFS: [1, 2]
chèn 3: cha 1; hàng đợi giờ: [2, 3]; cây [1, 2, 3]
chèn 4: cha 2; hàng đợi giờ: [2, 3, 4]; cây [1, 2, 3, 4]


## 5. Cách 1 — Brute Force: BFS tìm vị trí mỗi lần chèn

**Ý tưởng:** mỗi lần `insert(val)`, BFS từ gốc đến nút đầu tiên chưa đủ hai con; gắn nút mới vào con trái nếu thiếu, ngược lại con phải.

In [5]:
class CBTBrute:
    def __init__(self, root):
        self.root = root

    def insert(self, val):
        q = deque([self.root])
        while q:
            node = q.popleft()
            if node.left is None:                        # nút đầu tiên theo BFS còn thiếu con
                node.left = TreeNode(val)
                return node.val
            if node.right is None:
                node.right = TreeNode(val)
                return node.val
            q.append(node.left)
            q.append(node.right)

    def get_root(self):
        return self.root


assert run_ops(CBTBrute, example_ops, example_args) == example_out
print(run_ops(CBTBrute, example_ops, example_args))

[None, 1, 2, [1, 2, 3, 4]]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng mỗi insert O(n))
class CBTInserterBrute:

    def __init__(self, root: Optional[TreeNode]):
        self.root = root

    def insert(self, val: int) -> int:
        q = deque([self.root])
        while q:
            node = q.popleft()
            if node.left is None:
                node.left = TreeNode(val)
                return node.val
            if node.right is None:
                node.right = TreeNode(val)
                return node.val
            q.append(node.left)
            q.append(node.right)

    def get_root(self) -> Optional[TreeNode]:
        return self.root

**Độ phức tạp:** `insert` `O(n)` (BFS qua tới `n/2` nút đầu), `get_root` `O(1)`; với `10^4` lệnh và `n ≤ 1000 + 10^4` là `~10^8` bước trong trường hợp xấu. Bộ nhớ `O(n)` hàng đợi tạm.

**Nút thắt:** mỗi lần BFS lại duyệt phần đầu cây đã đầy. Nút "đầu tiên còn thiếu con" thay đổi rất chậm: chỉ cần nhớ nó.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ **hàng đợi** các nút còn thiếu con theo thứ tự BFS. `insert`: `parent = hàng đợi[0]`; gắn nút mới (trái nếu thiếu, ngược lại phải và bỏ `parent` khỏi hàng đợi); đẩy nút mới vào cuối. Hoặc lưu cây dạng **mảng nút theo tầng**: cha của nút mới ở chỉ số `(n − 1) // 2`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** hàng đợi các nút còn thiếu con, **Cách 3** mảng nút theo tầng, **Cách 4** đường đi theo bit của vị trí.

### Cách 2 — Hàng đợi các nút còn thiếu con

**Giải thích từng đoạn:**

1. Khởi tạo: BFS qua cây; đưa vào `deque` mọi nút có `left is None or right is None` theo thứ tự BFS.
2. `insert`: `parent = dq[0]`; nếu `parent.left is None` gắn con trái, ngược lại gắn con phải và `dq.popleft()`.
3. Đẩy nút mới vào cuối `dq`; trả `parent.val`.

In [7]:
class CBTQueue:
    def __init__(self, root):
        self.root = root
        self.dq = deque(n for n in all_nodes_bfs(root) if n.left is None or n.right is None)

    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node                           # gắn con trái: parent vẫn còn thiếu con phải, giữ lại hàng đợi
        else:
            parent.right = node
            self.dq.popleft()                            # parent đã đủ hai con
        self.dq.append(node)                             # nút mới chưa có con nào
        return parent.val

    def get_root(self):
        return self.root


assert run_ops(CBTQueue, example_ops, example_args) == example_out
print(run_ops(CBTQueue, example_ops, example_args))

[None, 1, 2, [1, 2, 3, 4]]


In [8]:
# Code nộp LeetCode cho Cách 2
class CBTInserterQueue:

    def __init__(self, root: Optional[TreeNode]):
        self.root = root
        self.dq = deque()
        q = deque([root])
        while q:
            node = q.popleft()
            if node.left is None or node.right is None:
                self.dq.append(node)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)

    def insert(self, val: int) -> int:
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node
        else:
            parent.right = node
            self.dq.popleft()
        self.dq.append(node)
        return parent.val

    def get_root(self) -> Optional[TreeNode]:
        return self.root

**Độ phức tạp:** khởi tạo `O(n)`; `insert` `O(1)`; `get_root` `O(1)`; bộ nhớ `O(n)` (tối đa `n/2 + 1` nút trong hàng đợi).

### Cách 3 — Mảng nút theo tầng

**Ý tưởng:** lưu `nodes` = danh sách các nút theo thứ tự BFS (chỉ số `0..n−1`). Nút mới có chỉ số `n`; `parent = nodes[(n − 1) // 2]`; nếu `n` lẻ gắn trái, ngược lại gắn phải. Thêm nút mới vào cuối `nodes`.

In [9]:
class CBTArray:
    def __init__(self, root):
        self.root = root
        self.nodes = all_nodes_bfs(root)                 # thứ tự BFS = chỉ số 0..n-1 của cây hoàn chỉnh

    def insert(self, val):
        n = len(self.nodes)
        parent = self.nodes[(n - 1) // 2]
        node = TreeNode(val)
        if n % 2 == 1:                                   # n = 2p + 1: con trái của p
            parent.left = node
        else:                                            # n = 2p + 2: con phải của p
            parent.right = node
        self.nodes.append(node)
        return parent.val

    def get_root(self):
        return self.root


assert run_ops(CBTArray, example_ops, example_args) == example_out
print(run_ops(CBTArray, example_ops, example_args))

[None, 1, 2, [1, 2, 3, 4]]


In [10]:
# Code nộp LeetCode cho Cách 3
class CBTInserterArray:

    def __init__(self, root: Optional[TreeNode]):
        self.root = root
        self.nodes = []
        q = deque([root])
        while q:
            node = q.popleft()
            self.nodes.append(node)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)

    def insert(self, val: int) -> int:
        n = len(self.nodes)
        parent = self.nodes[(n - 1) // 2]
        node = TreeNode(val)
        if n % 2 == 1:
            parent.left = node
        else:
            parent.right = node
        self.nodes.append(node)
        return parent.val

    def get_root(self) -> Optional[TreeNode]:
        return self.root

**Độ phức tạp:** khởi tạo `O(n)`; `insert` `O(1)`; bộ nhớ `O(n)` (mảng mọi nút). Mã đơn giản nhất.

### Cách 4 — Đường đi theo bit của vị trí

**Ý tưởng:** chỉ lưu `count` (số nút hiện có). Vị trí 1-indexed của nút mới là `k = count + 1`. Đường đi từ gốc là các bit của `k` sau bit `1` đầu: đi theo mọi bit trừ bit cuối để tới `parent`; bit cuối quyết định trái (`0`) hay phải (`1`). Không cần hàng đợi hay mảng.

In [11]:
class CBTPath:
    def __init__(self, root):
        self.root = root
        self.count = len(all_nodes_bfs(root))

    def insert(self, val):
        self.count += 1
        k = self.count                                    # vị trí 1-indexed của nút mới
        bits = bin(k)[3:]                                 # bỏ '0b1': các bit còn lại chỉ đường từ gốc
        node = self.root
        for b in bits[:-1]:                               # đi tới cha
            node = node.right if b == "1" else node.left
        new = TreeNode(val)
        if bits[-1] == "0":
            node.left = new
        else:
            node.right = new
        return node.val

    def get_root(self):
        return self.root


assert run_ops(CBTPath, example_ops, example_args) == example_out
print(run_ops(CBTPath, example_ops, example_args))

[None, 1, 2, [1, 2, 3, 4]]


In [12]:
# Code nộp LeetCode cho Cách 4
class CBTInserterPath:

    def __init__(self, root: Optional[TreeNode]):
        self.root = root
        self.count, q = 0, deque([root])
        while q:
            node = q.popleft()
            self.count += 1
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)

    def insert(self, val: int) -> int:
        self.count += 1
        bits = bin(self.count)[3:]
        node = self.root
        for b in bits[:-1]:
            node = node.right if b == "1" else node.left
        new = TreeNode(val)
        if bits[-1] == "0":
            node.left = new
        else:
            node.right = new
        return node.val

    def get_root(self) -> Optional[TreeNode]:
        return self.root

**Độ phức tạp:** khởi tạo `O(n)` (đếm); `insert` `O(log n)`; bộ nhớ `O(1)` thêm. Khởi tạo có thể nhanh hơn bằng cách đếm theo chiều cao (`O(log² n)`), xem bài 222.

## 8. TRACE từng bước

**Trace Cách 2** với cây `[1, 2]`, chèn `3`, `4`:

| lệnh | hàng đợi trước | `parent` | thao tác | hàng đợi sau | cây |
|---|---|---|---|---|---|
| khởi tạo | — | — | BFS: `1` (thiếu phải), `2` (không con) | `[1, 2]` | `[1,2]` |
| `insert(3)` | `[1, 2]` | `1` | `1.left` có, gắn `1.right = 3`, pop `1`, push `3` | `[2, 3]` | `[1,2,3]` |
| `insert(4)` | `[2, 3]` | `2` | `2.left` rỗng: gắn `2.left = 4`, push `4` | `[2, 3, 4]` | `[1,2,3,4]` |

Kết quả trả về `1`, `2` ✓. Hàm trace:

In [13]:
def trace_queue(init, inserts):
    root = build_tree(init)
    dq = deque(n for n in all_nodes_bfs(root) if n.left is None or n.right is None)
    print(f"cây ban đầu {to_level(root)}; hàng đợi (nút còn thiếu con theo BFS): {[n.val for n in dq]}")
    for v in inserts:
        parent = dq[0]
        node = TreeNode(v)
        side = "trái" if parent.left is None else "phải"
        if parent.left is None:
            parent.left = node
        else:
            parent.right = node
            dq.popleft()
        dq.append(node)
        print(f"insert({v}): cha = {parent.val}, gắn con {side}; hàng đợi = {[n.val for n in dq]}; cây = {to_level(root)}")
    return root


trace_queue([1, 2], [3, 4, 5, 6, 7])

cây ban đầu [1, 2]; hàng đợi (nút còn thiếu con theo BFS): [1, 2]
insert(3): cha = 1, gắn con phải; hàng đợi = [2, 3]; cây = [1, 2, 3]
insert(4): cha = 2, gắn con trái; hàng đợi = [2, 3, 4]; cây = [1, 2, 3, 4]
insert(5): cha = 2, gắn con phải; hàng đợi = [3, 4, 5]; cây = [1, 2, 3, 4, 5]
insert(6): cha = 3, gắn con trái; hàng đợi = [3, 4, 5, 6]; cây = [1, 2, 3, 4, 5, 6]
insert(7): cha = 3, gắn con phải; hàng đợi = [4, 5, 6, 7]; cây = [1, 2, 3, 4, 5, 6, 7]


**Trace Cách 4:** `count = 2`. `insert(3)`: `count = 3`, `bin(3) = '0b11'` ⇒ bits `'1'`; không có bit để đi (`bits[:-1]` rỗng), `node = root`; bit cuối `'1'` ⇒ con phải của gốc ✓. `insert(4)`: `count = 4`, `bin(4) = '0b100'` ⇒ bits `'00'`; đi `bits[:-1] = '0'` ⇒ trái (`2`); bit cuối `'0'` ⇒ con trái của `2` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 2** (hàng đợi, `O(1)` mỗi lệnh). Giữ nguyên tên lớp `CBTInserter` và các phương thức:

In [14]:
class CBTInserter:

    def __init__(self, root: Optional[TreeNode]):
        self.root = root
        self.dq = deque()                                 # các nút còn thiếu con, theo thứ tự BFS
        q = deque([root])
        while q:
            node = q.popleft()
            if node.left is None or node.right is None:
                self.dq.append(node)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)

    def insert(self, val: int) -> int:
        parent = self.dq[0]                               # nút đầu tiên còn thiếu con là cha của nút mới
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node                            # còn thiếu con trái (và phải): vẫn giữ parent trong hàng đợi
        else:
            parent.right = node
            self.dq.popleft()                             # đủ hai con: bỏ khỏi hàng đợi
        self.dq.append(node)                              # nút mới chưa có con
        return parent.val

    def get_root(self) -> Optional[TreeNode]:
        return self.root


print(run_ops(CBTInserter, example_ops, example_args))     # [None, 1, 2, [1, 2, 3, 4]]
assert run_ops(CBTInserter, example_ops, example_args) == example_out

[None, 1, 2, [1, 2, 3, 4]]


## 10. Vì sao đúng

**(a) Vị trí nút mới.** Cây hoàn chỉnh `n` nút ⇔ các chỉ số BFS `0..n−1` đều có mặt, không lỗ hổng. Để sau khi chèn vẫn hoàn chỉnh với `n + 1` nút, nút mới **phải** có chỉ số `n` (BFS tiếp theo); cha của nó là nút chỉ số `p = (n − 1) // 2` (vì con của `p` là `2p + 1`, `2p + 2` và `2p + 1 ≤ n`).

**(b) Nút "đầu tiên còn thiếu con".** Trong cây hoàn chỉnh, nút `i` thiếu con trái nếu `2i + 1 ≥ n`, thiếu con phải nếu `2i + 2 ≥ n`. Nút đầu tiên theo BFS thiếu con (`≥ 1` con) chính là nút `p = (n − 1) // 2`: với `i < p` có `2i + 2 ≤ 2p < n` (đủ hai con); nút `p` có `2p + 1 ≤ n`. Vậy `p` = cha đã nêu trong (a), và `n` lẻ ⇔ con trái chưa có, `n` chẵn ⇔ chỉ thiếu con phải.

**(c) Cách 2: bất biến hàng đợi.** Hàng đợi chứa **đúng** các nút còn thiếu con (≥ 1) theo thứ tự BFS. Khởi tạo: đúng theo định nghĩa. Mỗi lần `insert`: nút `parent` ở đầu. Nếu `parent.left is None` thì gắn trái: `parent` vẫn thiếu con phải ⇒ giữ nguyên trong hàng đợi (vẫn là đầu). Nếu chỉ thiếu phải: gắn phải ⇒ đủ hai con ⇒ bỏ khỏi đầu. Nút mới chưa có con nên thuộc tập "thiếu con" và có thứ tự BFS **sau cùng** ⇒ đẩy cuối. Bất biến được giữ.

**(d) Cách 3** là (a) áp dụng trực tiếp với mảng BFS.

**(e) Cách 4.** Nút BFS chỉ số `k − 1` (1-indexed `k`) có đường đi từ gốc là các bit của `k` sau bit `1` đầu: quy nạp theo độ sâu — bit đầu của phần còn lại cho biết con thứ nhất (`k` ↦ con trái `2k`, con phải `2k + 1`, tức thêm bit `0` hoặc `1` vào cuối biểu diễn nhị phân).

**(f) `get_root`.** Gốc không đổi (chèn chỉ thêm lá), nên trả cùng một đối tượng.

Ta kiểm chứng bằng máy: mọi cách khớp **mô hình mảng** (chèn = append) trên mọi kích thước ban đầu `1..40` và mọi số lần chèn `0..40`; cây luôn hoàn chỉnh sau mỗi bước (mục 12D), và khớp trên `10^4` lệnh ngẫu nhiên (12E–F).

## 11. Time / Space Complexity

| Cách | Khởi tạo | `insert` | Space | Ghi chú |
|---|---|---|---|---|
| 1. BFS mỗi lần | `O(1)` | `O(n)` | `O(n)` tạm | oracle thứ hai |
| 2. Hàng đợi nút thiếu con | `O(n)` | `O(1)` | `O(n)` | **đáp án chính** |
| 3. Mảng nút theo tầng | `O(n)` | `O(1)` | `O(n)` | đơn giản nhất |
| 4. Đường đi theo bit | `O(n)` (đếm) | `O(log n)` | `O(1)` | không cấu trúc phụ |

Đo thực tế (khởi tạo 1000 nút, rồi `9000` lệnh chèn; cách 1 chỉ `2000` lệnh):

In [15]:
def random_complete(rng, n):
    return build_tree([rng.randint(0, 5000) for _ in range(n)])         # danh sách đầy đủ theo tầng -> cây hoàn chỉnh


def bench(cls, n0, m, seed=0, repeat=3):
    best = 1e9
    for _ in range(repeat):
        rng = random.Random(seed)
        root = random_complete(rng, n0)
        t0 = time.perf_counter()
        obj = cls(root)
        for _ in range(m):
            obj.insert(rng.randint(0, 5000))
        best = min(best, time.perf_counter() - t0)
    return best


for name, cls in [("queue", CBTQueue), ("array", CBTArray), ("path", CBTPath)]:
    print(f"{name:6s}: khởi tạo 1000 + 9000 insert: {bench(cls, 1000, 9000) * 1e3:7.2f} ms")
print(f"brute : khởi tạo 1000 + 2000 insert: {bench(CBTBrute, 1000, 2000, repeat=1) * 1e3:7.1f} ms (mỗi insert O(n))")

queue : khởi tạo 1000 + 9000 insert:    5.84 ms
array : khởi tạo 1000 + 9000 insert:    6.32 ms
path  : khởi tạo 1000 + 9000 insert:   10.71 ms


brute : khởi tạo 1000 + 2000 insert:   169.7 ms (mỗi insert O(n))


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề (qua `run_ops`), (B) edge case, (C) phản ví dụ, (D) vét cạn mọi kích thước ban đầu và số lần chèn với oracle mô hình mảng, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn `10^4` lệnh.

### A. Ví dụ của đề

In [16]:
class CBTModel:
    """Oracle: mô hình MẢNG theo tầng của cây hoàn chỉnh (chèn = append, cha của chỉ số k là (k-1)//2); get_root = chính mảng."""
    def __init__(self, root):
        self.vals = to_level(root)

    def insert(self, val):
        self.vals.append(val)
        return self.vals[(len(self.vals) - 2) // 2]       # chỉ số nút mới là len-1; cha là (len-1-1)//2

    def get_root(self):
        return list(self.vals)


CLASSES = {
    "brute": CBTBrute,
    "queue": CBTQueue,
    "array": CBTArray,
    "path": CBTPath,
    "CBTInserter": CBTInserter,
    "CBTInserterBrute": CBTInserterBrute,
    "CBTInserterQueue": CBTInserterQueue,
    "CBTInserterArray": CBTInserterArray,
    "CBTInserterPath": CBTInserterPath,
}


def check_script(init, script, names=None):
    """script: danh sách các lệnh ('insert', v) hoặc ('get_root',). So với mô hình mảng; kiểm tra tính hoàn chỉnh sau mỗi bước."""
    for name in (names or CLASSES):
        root = build_tree(init)
        obj = CLASSES[name](root)
        model = CBTModel(build_tree(init))
        for cmd in script:
            if cmd[0] == "insert":
                got, want = obj.insert(cmd[1]), model.insert(cmd[1])
                assert got == want, (name, init, script, cmd, got, want)
            else:
                got = obj.get_root()
                assert got is root and to_level(got) == model.get_root(), (name, init, script)
            if len(model.vals) <= 80:
                assert is_complete(root), (name, init, script)
        assert is_complete(root), (name, init, script)


for cls in CLASSES.values():
    assert run_ops(cls, example_ops, example_args) == example_out
check_script([1, 2], [("insert", 3), ("insert", 4), ("get_root",)])
print("ví dụ đề: mô hình mảng và", len(CLASSES), "cài đặt (chạy bằng run_ops) đều cho", example_out)

ví dụ đề: mô hình mảng và 9 cài đặt (chạy bằng run_ops) đều cho [None, 1, 2, [1, 2, 3, 4]]


### B. Edge case

- Cây một nút: chèn đầu tiên gắn con trái của gốc, tiếp theo con phải.
- Cây đầy đủ `2^h − 1` nút: nút mới mở tầng mới (con trái của nút ngoài cùng bên trái).
- Cây thiếu đúng một nút ở tầng cuối: chèn lấp đầy tầng.
- Nhiều lệnh `get_root` liên tiếp (không đổi gì).
- Chèn rất nhiều nút liên tiếp (nhiều tầng mới).
- Giá trị `0` và `5000`.
- `get_root` trả cùng một đối tượng gốc (`is`).

In [17]:
check_script([7], [("insert", 8), ("insert", 9), ("insert", 10), ("get_root",)])
check_script([1, 2, 3], [("insert", 4), ("insert", 5), ("insert", 6), ("insert", 7), ("insert", 8), ("get_root",)])         # đầy -> tầng mới
check_script([1, 2, 3, 4, 5, 6], [("insert", 7), ("insert", 8), ("get_root",), ("get_root",)])
check_script([0], [("insert", 5000)] * 10 + [("get_root",)])
check_script([5000, 0, 5000, 0], [("insert", 0), ("insert", 5000), ("get_root",)])
obj = CBTInserter(build_tree([1]))
r = obj.get_root()
obj.insert(2)
assert obj.get_root() is r
assert run_ops(CBTInserter, ["CBTInserter", "insert", "insert", "insert", "get_root"], [[[1]], [2], [3], [4], []]) == [None, 1, 1, 2, [1, 2, 3, 4]]
print("edge case: đúng, cây luôn hoàn chỉnh, get_root trả cùng đối tượng")

edge case: đúng, cây luôn hoàn chỉnh, get_root trả cùng đối tượng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
class BugPopOnLeft(CBTQueue):                          # bỏ nút khỏi hàng đợi ngay sau khi gắn con TRÁI
    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node
            self.dq.popleft()                          # SAI: parent vẫn thiếu con phải
        else:
            parent.right = node
            self.dq.popleft()
        self.dq.append(node)
        return parent.val


class BugNoPop(CBTQueue):                              # không bao giờ bỏ nút khỏi hàng đợi
    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node
        else:
            parent.right = node                        # SAI: lần sau lại thế nút này (ghi đè con phải)
        self.dq.append(node)
        return parent.val


class BugNoPush(CBTQueue):                             # quên đẩy nút mới vào hàng đợi
    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node
        else:
            parent.right = node
            self.dq.popleft()
        return parent.val                              # SAI: thiếu self.dq.append(node)


class BugInitAll(CBTQueue):                            # hàng đợi ban đầu chứa mọi nút (kể cả đã đủ hai con)
    def __init__(self, root):
        self.root = root
        self.dq = deque(all_nodes_bfs(root))


class BugPathOffByOne(CBTPath):                        # đi theo bit sai: dùng bits[1:] của vị trí 0-indexed
    def insert(self, val):
        k = self.count                                 # SAI: chưa tăng đếm, dùng k = count thay vì count + 1
        self.count += 1
        bits = bin(k + 0)[3:] if k > 1 else ""
        node = self.root
        for b in bits[:-1]:
            node = node.right if b == "1" else node.left
        new = TreeNode(val)
        if bits and bits[-1] == "0":
            node.left = new
        else:
            node.right = new
        return node.val


class BugReturnNode(CBTQueue):                         # trả nút cha thay vì giá trị
    def insert(self, val):
        parent = self.dq[0]
        super().insert(val)
        return parent


BUGS = [BugPopOnLeft, BugNoPop, BugNoPush, BugInitAll, BugReturnNode]
rng = random.Random(3)
for bug in BUGS:
    found = None
    for n0 in range(1, 8):
        init = list(range(100, 100 + n0))
        script = [("insert", 200 + i) for i in range(8)]
        root = build_tree(init)
        obj, model = bug(root), CBTModel(build_tree(init))
        try:
            for _, v in script:
                got, want = obj.insert(v), model.insert(v)
                if got != want or not is_complete(root) or to_level(root) != model.get_root():
                    found = (init, v, got, want, to_level(root), model.get_root())
                    break
        except (AttributeError, IndexError) as e:
            found = (init, "ngoại lệ", type(e).__name__)
        if found:
            break
    assert found is not None, bug.__name__
    print(f"{bug.__name__:15s} sai: {found[:4]}...")

BugPopOnLeft    sai: ([100], 201, 200, 100)...
BugNoPop        sai: ([100], 202, 100, 200)...
BugNoPush       sai: ([100], 'ngoại lệ', 'IndexError')...
BugInitAll      sai: ([100, 101, 102], 200, 100, 101)...
BugReturnNode   sai: ([100], 200, <__main__.TreeNode object at 0x7ff5e80d17f0>, 100)...


### D. Vét cạn mọi kích thước ban đầu và số lần chèn

Mọi kích thước ban đầu `n0 ∈ [1, 40]` và mọi số lần chèn `m ∈ [0, 40]` (`1 640` kịch bản), giá trị **khác nhau** (để nhận diện đúng nút cha), xen kẽ `get_root` sau mỗi lần chèn. Oracle: mô hình mảng. Sau mỗi bước cây phải **hoàn chỉnh**.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "queue", "array", "path", "CBTInserter")
for n0 in range(1, 41):
    init = list(range(1, n0 + 1))
    for m in range(0, 41):
        script = []
        for i in range(m):
            script += [("insert", 1000 + i), ("get_root",)]
        check_script(init, script, names=fast)
        cnt += 1
print(cnt, "kịch bản: 5 cài đặt khớp mô hình mảng, cây luôn hoàn chỉnh;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 40 * 41

1640 kịch bản: 5 cài đặt khớp mô hình mảng, cây luôn hoàn chỉnh; 2.8 s


### E. Ngẫu nhiên có seed và tính chất

Cây ban đầu ngẫu nhiên (`1..300` nút, **giá trị có thể trùng**), chuỗi lệnh ngẫu nhiên tới `150` lệnh. Tính chất: (1) sau mọi lệnh cây hoàn chỉnh; (2) số nút tăng đúng `1` mỗi `insert`; (3) mọi lá mới nằm ở tầng cuối hoặc mở tầng mới đúng quy tắc (độ cao cây `= ⌊log₂ n⌋ + 1`); (4) kết quả `insert` luôn là giá trị của nút có chỉ số BFS `(n−1)//2`.

In [20]:
rng = random.Random(20240919)
for _ in range(250):
    n0 = rng.randint(1, 300)
    init = [rng.randint(0, 5000) for _ in range(n0)]
    script = []
    for _ in range(rng.randint(0, 150)):
        script.append(("insert", rng.randint(0, 5000)) if rng.random() < 0.8 else ("get_root",))
    check_script(init, script, names=("queue", "array", "path", "CBTInserter", "CBTInserterQueue", "CBTInserterArray", "CBTInserterPath"))
    root = build_tree(init)
    obj = CBTInserter(root)
    n = n0
    for cmd in script:
        if cmd[0] == "insert":
            nodes = all_nodes_bfs(root)
            expect_parent = nodes[(len(nodes) - 1) // 2].val                          # (4)
            assert obj.insert(cmd[1]) == expect_parent
            n += 1
            assert len(all_nodes_bfs(root)) == n                                       # (2)
    height = 0
    q = deque([(root, 1)])
    while q:
        node, d = q.popleft()
        height = max(height, d)
        for ch in (node.left, node.right):
            if ch:
                q.append((ch, d + 1))
    assert height == n.bit_length()                                                    # (3): chiều cao = floor(log2 n) + 1
print("250 kịch bản ngẫu nhiên: 7 cài đặt khớp mô hình mảng; 4 tính chất (hoàn chỉnh, +1 nút, chiều cao, cha) đúng")

250 kịch bản ngẫu nhiên: 7 cài đặt khớp mô hình mảng; 4 tính chất (hoàn chỉnh, +1 nút, chiều cao, cha) đúng


### F. Giới hạn: `10^4` lệnh

Khởi tạo 1000 nút, `9000` lệnh chèn (tổng `10^4` lệnh với `get_root` xen kẽ): 4 cách nhanh khớp nhau và khớp mô hình; cây cuối cùng có `10 000` nút và hoàn chỉnh. Cách 1 kiểm tra trên 1500 lệnh đầu.

In [21]:
rng = random.Random(500)
init = [rng.randint(0, 5000) for _ in range(1000)]
script = []
for i in range(9000):
    script.append(("insert", rng.randint(0, 5000)))
    if i % 9 == 0:
        script.append(("get_root",))
check_script(init, script, names=("queue", "array", "path", "CBTInserter", "CBTInserterQueue", "CBTInserterArray", "CBTInserterPath"))
check_script(init, script[:1500], names=("brute",))
root = build_tree(init)
obj = CBTInserter(root)
for cmd in script:
    if cmd[0] == "insert":
        obj.insert(cmd[1])
assert len(all_nodes_bfs(root)) == 10000 and is_complete(root)
print("10^4 lệnh: 7 cài đặt nhanh khớp mô hình; cây cuối 10 000 nút, hoàn chỉnh; cách 1 khớp 1500 lệnh đầu")

10^4 lệnh: 7 cài đặt nhanh khớp mô hình; cây cuối 10 000 nút, hoàn chỉnh; cách 1 khớp 1500 lệnh đầu


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ nút khỏi hàng đợi ngay sau khi gắn con **trái**

In [22]:
root = build_tree([1])
obj = BugPopOnLeft(root)
print([obj.insert(v) for v in (2, 3, 4)], "| đúng:", [1, 1, 2])
assert [BugPopOnLeft(build_tree([1])).insert(2)] == [1]
o = BugPopOnLeft(build_tree([1]))
assert [o.insert(v) for v in (2, 3, 4)] != [1, 1, 2]

[1, 2, 3] | đúng: [1, 1, 2]


**Vì sao sai:** sau khi gắn con trái, nút vẫn **thiếu con phải** nên phải ở lại hàng đợi. **Đúng:** chỉ `popleft` sau khi gắn con **phải**.

### Lỗi 2: không bao giờ bỏ nút khỏi hàng đợi

In [23]:
o = BugNoPop(build_tree([1]))
root = o.root
print([o.insert(v) for v in (2, 3, 4)], to_level(root), "| đúng cây: [1, 2, 3, 4]")
assert to_level(root) != [1, 2, 3, 4]

[1, 1, 1] [1, 2, 4] | đúng cây: [1, 2, 3, 4]


**Vì sao sai:** sau khi đủ hai con, nút phải rời hàng đợi; nếu không các lần sau ghi đè con phải. **Đúng:** `popleft`.

### Lỗi 3: quên đẩy nút mới vào hàng đợi

In [24]:
o = BugNoPush(build_tree([1]))
try:
    print([o.insert(v) for v in (2, 3, 4)])
except IndexError as e:
    print("IndexError:", e, "(hàng đợi cạn: không còn nút nào để gắn)")

IndexError: deque index out of range (hàng đợi cạn: không còn nút nào để gắn)


**Vì sao sai:** nút mới là ứng viên để nhận con ở các lần chèn sau. **Đúng:** `self.dq.append(node)`.

### Lỗi 4: khởi tạo hàng đợi bằng mọi nút

In [25]:
init = [1, 2, 3]
o = BugInitAll(build_tree(init))
r = o.root
print("chèn 4:", o.insert(4), "| cây:", to_level(r), "| đúng: cha 2, cây [1,2,3,4]")
assert to_level(r) != [1, 2, 3, 4]

chèn 4: 1 | cây: [1, 2, 4] | đúng: cha 2, cây [1,2,3,4]


**Vì sao sai:** hàng đợi chỉ được chứa nút **còn thiếu con**; gắn nút mới vào nút đã đủ hai con làm ghi đè. **Đúng:** lọc `left is None or right is None`.

### Lỗi 5: trả về nút cha thay vì giá trị

In [26]:
o = BugReturnNode(build_tree([1, 2]))
res = o.insert(3)
print("kết quả:", type(res).__name__, "| đúng: int", CBTInserter(build_tree([1, 2])).insert(3))
assert isinstance(res, TreeNode)

kết quả: TreeNode | đúng: int 1


**Vì sao sai:** đề yêu cầu trả **giá trị** (`int`) của nút cha. **Đúng:** `return parent.val`.

### Lỗi 6 (bonus): đường đi theo bit lệch một

Dùng `count` chưa tăng hoặc `bits` tính từ `k − 1`: nút mới gắn sai chỗ. Hãy luôn tăng `count` trước, dùng `bin(count)[3:]`.

In [27]:
for k in range(2, 9):
    bits = bin(k)[3:]
    print(f"vị trí {k}: bin = {bin(k)[2:]:>4}, bits sau bit 1 đầu = {bits!r:>5} -> đường từ gốc {''.join('R' if b == '1' else 'L' for b in bits)}")
print("lệch một: dùng bin(k - 1) hoặc quên tăng count trước khi tính sẽ cho đường đi sai, ví dụ vị trí 5 -> bits", bin(5)[3:], "nhưng bin(4)[3:] =", bin(4)[3:])

vị trí 2: bin =   10, bits sau bit 1 đầu =   '0' -> đường từ gốc L
vị trí 3: bin =   11, bits sau bit 1 đầu =   '1' -> đường từ gốc R
vị trí 4: bin =  100, bits sau bit 1 đầu =  '00' -> đường từ gốc LL
vị trí 5: bin =  101, bits sau bit 1 đầu =  '01' -> đường từ gốc LR
vị trí 6: bin =  110, bits sau bit 1 đầu =  '10' -> đường từ gốc RL
vị trí 7: bin =  111, bits sau bit 1 đầu =  '11' -> đường từ gốc RR
vị trí 8: bin = 1000, bits sau bit 1 đầu = '000' -> đường từ gốc LLL
lệch một: dùng bin(k - 1) hoặc quên tăng count trước khi tính sẽ cho đường đi sai, ví dụ vị trí 5 -> bits 01 nhưng bin(4)[3:] = 00


**Vì sao quan trọng:** bit đầu tiên sau bit `1` chỉ nhánh từ gốc; bit cuối chỉ nút mới là con trái (`0`) hay phải (`1`) của cha.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "cây **hoàn chỉnh** + thao tác thêm vào cuối" → cây hoàn chỉnh tương đương **mảng theo tầng** (heap indexing): vị trí, cha, con đều tính bằng số học. Thiết kế lớp: tìm trạng thái nhỏ nhất đủ để trả lời mọi lệnh.

**Mẹo:**

1. Cha của chỉ số `i` (0-indexed) là `(i − 1) // 2`; con `2i + 1`, `2i + 2`.
2. Nút mới luôn ở chỉ số `n`; trái nếu `n` lẻ, phải nếu `n` chẵn.
3. Hàng đợi các nút thiếu con thay đổi chỉ ở đầu (pop) và cuối (push).
4. Nhớ đẩy nút mới vào hàng đợi và pop đúng lúc (sau khi gắn con phải).
5. Test thiết kế lớp bằng bộ chạy chuỗi lệnh + mô hình mảng làm oracle + kiểm tra tính hoàn chỉnh.
6. Biểu diễn bit của vị trí cho đường đi `O(log n)` không cần cấu trúc phụ.

**Họ bài cùng pattern:** Count Complete Tree Nodes (222), Check Completeness of a Binary Tree (958), Serialize and Deserialize Binary Tree (297), Binary Heap (đống), Kth Largest Element in a Stream (703), Design Underground System.

## 15. Biến thể

1. **`remove_last()`** — xoá nút cuối cùng, giữ cây hoàn chỉnh.
2. **`size()`** — số nút (đã có `count`).
3. **`get(k)`** — giá trị nút thứ `k` theo thứ tự tầng, bằng đường đi theo bit.
4. **Kiểm tra cây hoàn chỉnh** (958).
5. **Đếm nút cây hoàn chỉnh** (222) theo chiều cao.

Cài biến thể 1–3 trên Cách 3 và đối chiếu mô hình mảng:

In [28]:
class CBTExtended(CBTArray):
    def remove_last(self):
        """Xoá nút cuối cùng (chỉ số n-1), giữ cây hoàn chỉnh; trả giá trị của nó. Không xoá nếu chỉ còn gốc."""
        if len(self.nodes) == 1:
            return None
        node = self.nodes.pop()
        parent = self.nodes[(len(self.nodes) - 1) // 2]
        if len(self.nodes) % 2 == 0:                     # nút bị xoá có chỉ số n-1 (n cũ): sau pop độ dài = n-1
            parent.right = None
        else:
            parent.left = None
        return node.val

    def size(self):
        return len(self.nodes)

    def get(self, k):
        """Giá trị nút thứ k (1-indexed theo tầng) bằng đường đi theo bit (không dùng mảng nodes)."""
        node = self.root
        for b in bin(k)[3:]:
            node = node.right if b == "1" else node.left
        return node.val


def count_complete(root):
    """Biến thể 5 (222): đếm nút cây hoàn chỉnh O(log^2 n) theo chiều cao nhánh trái/phải."""
    if root is None:
        return 0
    hl = hr = 0
    a = b = root
    while a:
        hl, a = hl + 1, a.left
    while b:
        hr, b = hr + 1, b.right
    if hl == hr:
        return (1 << hl) - 1
    return 1 + count_complete(root.left) + count_complete(root.right)


rng = random.Random(15)
for _ in range(300):
    n0 = rng.randint(1, 30)
    root = build_tree([rng.randint(0, 50) for _ in range(n0)])
    obj = CBTExtended(root)
    model = to_level(root)
    for _ in range(rng.randint(1, 30)):
        if rng.random() < 0.6 or len(model) == 1:
            v = rng.randint(0, 50)
            assert obj.insert(v) == model[(len(model) - 1) // 2]
            model.append(v)
        else:
            assert obj.remove_last() == model.pop()
        assert to_level(root) == model and is_complete(root)
        assert obj.size() == len(model) == count_complete(root)
        k = rng.randint(1, len(model))
        assert obj.get(k) == model[k - 1]
print("remove_last, size, get(k) theo bit, đếm nút 222: khớp mô hình mảng; cây luôn hoàn chỉnh (300 kịch bản)")

remove_last, size, get(k) theo bit, đếm nút 222: khớp mô hình mảng; cây luôn hoàn chỉnh (300 kịch bản)


## 16. So sánh các cách

| Cách | Khởi tạo | `insert` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mỗi lần | `O(1)` | `O(n)` | `O(n)` tạm | oracle thứ hai |
| 2. Hàng đợi nút thiếu con | `O(n)` | `O(1)` | `O(n)` | **đáp án chính** |
| 3. Mảng nút theo tầng | `O(n)` | `O(1)` | `O(n)` | đơn giản nhất |
| 4. Đường đi theo bit | `O(n)` | `O(log n)` | `O(1)` | không cấu trúc phụ |

Cả bốn khớp mô hình mảng trên 1 640 kịch bản vét cạn, 250 kịch bản ngẫu nhiên và `10^4` lệnh (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(run_ops(CBTInserter, ["CBTInserter", "insert", "insert", "insert", "insert", "get_root"], [[[5]], [6], [7], [8], [9], []]))

[None, 5, 5, 6, 6, [5, 6, 7, 8, 9]]


### Bài 2 (tính tay)

Cây `[1, 2, 3, 4, 5, 6]` rồi lần lượt `insert(7)`, `insert(8)`, `insert(9)`. (a) Cha của mỗi nút mới? (b) Cây cuối? (c) Hàng đợi các nút thiếu con ban đầu? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện lớp với hàng đợi:

In [30]:
class CBTTodo:
    def __init__(self, root):
        self.root = root
        self.dq = deque()
        # TODO 1: BFS, đưa vào self.dq mọi nút còn thiếu con theo thứ tự BFS

    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        # TODO 2: gắn con trái nếu thiếu, ngược lại gắn con phải và popleft
        # TODO 3: đẩy nút mới vào cuối hàng đợi
        return parent.val

    def get_root(self):
        return self.root


# Khi làm xong, bỏ comment để kiểm tra:
# assert run_ops(CBTTodo, example_ops, example_args) == example_out

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MyInserter` bằng **đường đi theo bit** (`count` + `bin`):

In [31]:
class MyInserter:
    def __init__(self, root):
        pass            # TODO

    def insert(self, val):
        pass            # TODO

    def get_root(self):
        pass            # TODO


# Khi làm xong:
# assert run_ops(MyInserter, example_ops, example_args) == example_out

### Bài 5 (mini challenge): Check Completeness of a Binary Tree (958)

Viết `check_complete(root)` trả `True` nếu cây hoàn chỉnh (BFS có `None`; sau `None` đầu tiên không còn nút thật). Kiểm tra với cây sinh bởi `CBTInserter` (luôn hoàn chỉnh) và cây bị phá (gắn nút vào chỗ trống sai).

In [32]:
def check_complete(root):
    pass            # TODO


# Khi làm xong:
# assert check_complete(build_tree([1, 2, 3, 4, 5, 6])) and not check_complete(build_tree([1, 2, 3, 4, None, 6, 7]))

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: cây `[5]`; nút mới lần lượt ở chỉ số `1, 2, 3, 4`.
- Mức 2: cha của chỉ số `k` là `(k-1)//2`: `0, 0, 1, 1` → giá trị `5, 5, 6, 6`.
- Mức 3: `[None, 5, 5, 6, 6, [5, 6, 7, 8, 9]]`.

**Bài 2**

- Mức 1: cây có `6` nút; nút `7` có chỉ số `6`: cha chỉ số `(6-1)//2 = 2` (giá trị `3`).
- Mức 2: nút `8` chỉ số `7`: cha chỉ số `3` (giá trị `4`); nút `9` chỉ số `8`: cha chỉ số `3`.
- Mức 3: cha lần lượt `3, 4, 4`; cây `[1,2,3,4,5,6,7,8,9]`; hàng đợi ban đầu: nút `3` (thiếu con phải), `4`, `5`, `6`.

**Bài 3**

- Mức 1: BFS bằng `deque`.
- Mức 2: `if node.left is None or node.right is None: self.dq.append(node)`.
- Mức 3: xem Cách 2.

**Bài 4**

- Mức 1: đếm số nút trong `__init__`.
- Mức 2: `bits = bin(count)[3:]` sau khi tăng.
- Mức 3: đi theo `bits[:-1]`, gắn theo `bits[-1]`.

**Bài 5**

- Mức 1: BFS đẩy cả `None`.
- Mức 2: đặt cờ khi gặp `None`.
- Mức 3: gặp nút thật sau cờ → `False`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
out = run_ops(CBTInserter, ["CBTInserter", "insert", "insert", "insert", "insert", "get_root"], [[[5]], [6], [7], [8], [9], []])
assert out == [None, 5, 5, 6, 6, [5, 6, 7, 8, 9]]
print("Bài 1 OK:", out)

Bài 1 OK: [None, 5, 5, 6, 6, [5, 6, 7, 8, 9]]


### Bài 2

(a) cha `3, 4, 4`. (b) `[1,2,3,4,5,6,7,8,9]`. (c) hàng đợi ban đầu (nút còn thiếu con): `3` (thiếu con phải), `4`, `5`, `6`.

In [34]:
root = build_tree([1, 2, 3, 4, 5, 6])
obj = CBTInserter(root)
assert [n.val for n in obj.dq] == [3, 4, 5, 6]
assert [obj.insert(v) for v in (7, 8, 9)] == [3, 4, 4] and to_level(root) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
class CBTAnswer:
    def __init__(self, root):
        self.root = root
        self.dq = deque()
        q = deque([root])
        while q:
            node = q.popleft()
            if node.left is None or node.right is None:
                self.dq.append(node)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)

    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node
        else:
            parent.right = node
            self.dq.popleft()
        self.dq.append(node)
        return parent.val

    def get_root(self):
        return self.root


assert run_ops(CBTAnswer, example_ops, example_args) == example_out
check_script([1, 2, 3, 4], [("insert", i) for i in range(10, 25)], names=["queue"])
rng = random.Random(3)
for _ in range(100):
    init = [rng.randint(0, 50) for _ in range(rng.randint(1, 20))]
    root = build_tree(init)
    obj, model = CBTAnswer(root), CBTModel(build_tree(init))
    for _ in range(20):
        v = rng.randint(0, 50)
        assert obj.insert(v) == model.insert(v)
    assert to_level(root) == model.get_root()
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MyInserterAnswer:
    def __init__(self, root):
        self.root = root
        self.count = len(all_nodes_bfs(root))

    def insert(self, val):
        self.count += 1
        bits = bin(self.count)[3:]
        node = self.root
        for b in bits[:-1]:
            node = node.right if b == "1" else node.left
        new = TreeNode(val)
        if bits[-1] == "0":
            node.left = new
        else:
            node.right = new
        return node.val

    def get_root(self):
        return self.root


assert run_ops(MyInserterAnswer, example_ops, example_args) == example_out
rng = random.Random(4)
for _ in range(100):
    init = [rng.randint(0, 50) for _ in range(rng.randint(1, 20))]
    root = build_tree(init)
    obj, model = MyInserterAnswer(root), CBTModel(build_tree(init))
    for _ in range(20):
        v = rng.randint(0, 50)
        assert obj.insert(v) == model.insert(v)
    assert to_level(root) == model.get_root() and is_complete(root)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def check_complete_answer(root):
    seen_none = False
    q = deque([root])
    while q:
        node = q.popleft()
        if node is None:
            seen_none = True
        else:
            if seen_none:
                return False
            q.append(node.left)
            q.append(node.right)
    return True


assert check_complete_answer(build_tree([1, 2, 3, 4, 5, 6])) and not check_complete_answer(build_tree([1, 2, 3, 4, None, 6, 7]))
rng = random.Random(5)
for _ in range(200):
    root = build_tree([rng.randint(0, 9) for _ in range(rng.randint(1, 15))])
    assert check_complete_answer(root)                                          # cây dựng từ danh sách đầy đủ luôn hoàn chỉnh
    nodes = all_nodes_bfs(root)
    holes = [n for n in nodes if n.left is None and n.right is None]            # phá: gắn con phải vào lá không có con trái
    if holes:
        broken = rng.choice(holes)
        broken.right = TreeNode(99)
        assert not check_complete_answer(root) and not is_complete(root)
print("Bài 5 OK: nhận diện đúng cây hoàn chỉnh và cây bị phá")

Bài 5 OK: nhận diện đúng cây hoàn chỉnh và cây bị phá


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chèn vào cây hoàn chỉnh → hàng đợi các nút còn thiếu con):**

```python
class CBTInserter:
    def __init__(self, root):
        self.root = root
        self.dq = deque()
        q = deque([root])
        while q:
            node = q.popleft()
            if node.left is None or node.right is None:
                self.dq.append(node)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)

    def insert(self, val):
        parent = self.dq[0]
        node = TreeNode(val)
        if parent.left is None:
            parent.left = node
        else:
            parent.right = node
            self.dq.popleft()
        self.dq.append(node)
        return parent.val

    def get_root(self):
        return self.root
```

**Checklist trước khi submit:**

- [ ] Hàng đợi chỉ chứa nút **còn thiếu con**, theo thứ tự BFS?
- [ ] `popleft` **chỉ** sau khi gắn con **phải**?
- [ ] Đẩy nút mới vào **cuối** hàng đợi?
- [ ] `insert` trả **giá trị** của cha (`int`), `get_root` trả gốc (cùng đối tượng)?
- [ ] Khởi tạo bằng BFS đúng thứ tự trái → phải, từng tầng?
- [ ] Đã thử ví dụ `[1,2]` chèn `3, 4`, cây một nút, cây đầy đủ `2^h − 1`, nhiều lần chèn mở tầng mới?
- [ ] Đúng tên `class CBTInserter`, `insert(self, val)`, `get_root(self)`, không in debug.